# 02 — Limpieza: reglas de negocio verificables

## Pregunta que guía el notebook

Qué registros representan compras cuando queremos estimar una futura recompra. Limpiar no es borrar valores que se ven raros: es declarar reglas, medir su efecto y comprobar que se cumplieron.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
def find_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/retail_customer_intelligence')]:
        if (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto. Abre la carpeta retail_customer_intelligence, no sólo el archivo .ipynb.')
ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from retail_pipeline import *
ensure_folders(ROOT)
print('Proyecto:', ROOT)

## Decidir antes de transformar

Cada regla se conecta con la pregunta de negocio. El archivo original se conserva sin modificar; la tabla limpia es una vista reproducible para el objetivo de modelado.

In [ ]:
raw_path = ROOT / "data/interim/transacciones_originales.parquet"
if not raw_path.exists():
    raise FileNotFoundError("Ejecuta primero el notebook 01.")
raw = pd.read_parquet(raw_path)
clean, audit = clean_transactions(raw)
clean.to_parquet(ROOT / "data/processed/transacciones_limpias.parquet", index=False)
audit.to_csv(ROOT / "reports/auditoria_limpieza.csv", index=False)

rules = pd.DataFrame([
    ("CustomerID presente", "Sin identificador no existe un perfil de cliente", "excluir de ventas modelables"),
    ("Factura no cancelada", "Una cancelación no es una compra nueva", "excluir de ventas modelables"),
    ("Cantidad y precio positivos", "La etiqueta futura representa una compra", "excluir de ventas modelables"),
    ("Duplicado exacto", "Evita contar la misma línea dos veces", "conservar una copia"),
    ("Código de producto", "Un servicio no describe diversidad de artículos", "conservar gasto; no usar en diversidad"),
], columns=["regla", "por qué", "decisión"])
display(rules)

## Verificar la limpieza, no sólo ejecutarla

Las reglas se transforman en pruebas. Si una fuente futura deja pasar registros que contradicen estos supuestos, el pipeline falla en vez de entrenar silenciosamente con datos inconsistentes.

In [ ]:
checks = pd.DataFrame({
    "condición esperada": [
        "CustomerID presente", "cantidad positiva", "precio positivo",
        "sin facturas canceladas", "sin duplicados exactos",
    ],
    "incumplimientos": [
        clean.CustomerID.isna().sum(),
        clean.Quantity.le(0).sum(),
        clean.UnitPrice.le(0).sum(),
        clean.InvoiceNo.astype("string").str.startswith("C", na=False).sum(),
        clean.duplicated().sum(),
    ],
})
display(checks)
assert checks.incumplimientos.sum() == 0

summary = pd.DataFrame({
    "etapa": ["archivo original", "ventas válidas"],
    "líneas": [len(raw), len(clean)],
    "facturas": [raw.InvoiceNo.nunique(), clean.InvoiceNo.nunique()],
    "clientes": [raw.CustomerID.nunique(), clean.CustomerID.nunique()],
    "importe": [
        (raw.loc[raw.Quantity.gt(0) & raw.UnitPrice.gt(0), "Quantity"]
         * raw.loc[raw.Quantity.gt(0) & raw.UnitPrice.gt(0), "UnitPrice"]).sum(),
        clean.line_total.sum(),
    ],
})
display(summary.round(2))

## Qué se conserva y por qué

Las devoluciones y los códigos especiales no se eliminan del conocimiento del negocio. La pregunta sólo exige que no se cuenten como nuevas ventas. Más adelante podrán resumirse como una tasa de devolución o excluirse de una medida de variedad.

In [ ]:
special = clean.groupby("es_producto").agg(
    líneas=("InvoiceNo", "size"),
    importe=("line_total", "sum"),
    códigos=("StockCode", "nunique"),
).rename(index={True: "producto identificable", False: "servicio/código especial"})
display(special.round(2))

raw = raw.copy()
raw["InvoiceNo"] = raw.InvoiceNo.astype("string")
cancelled = raw.InvoiceNo.str.startswith("C", na=False)
examples = pd.concat([
    raw[cancelled].head(2).assign(tipo="cancelación"),
    raw[raw.Quantity.lt(0) & ~cancelled].head(2).assign(tipo="devolución/ajuste"),
    clean.head(2).assign(tipo="venta válida"),
])
display(examples[["tipo", "InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID", "InvoiceDate"]])

## Mensaje para clase

La limpieza no mejora una métrica por sí sola. Define qué hechos cuentan como ventas y deja una bitácora de por qué. Sin esa trazabilidad, una diferencia de modelo no puede defenderse.

Artefactos: data/processed/transacciones_limpias.parquet y reports/auditoria_limpieza.csv.